
# Multi‑turn Conversations with **Responses API** (Azure OpenAI)

This notebook demonstrates how to build a **multi‑turn conversation** using the `openai` Python package with **Azure OpenAI**.  
We’ll start from a single turn and then chain turns together by referencing the **previous response**—the supported way to maintain context on Azure—so you can progressively compose assistant behavior without manually replaying the entire history each time.

---

## What you will learn

- Load environment variables and initialize the `AzureOpenAI` client  
- Send a first `responses.create()` call and read its output  
- **Continue the conversation** by linking turns via `previous_response_id`  
- Compare with the stateless approach (managing message history yourself)  
- Practical tips: persistence, token budgeting, and error handling

---

## Prerequisites

- Python 3.10+  
- `openai` package installed (`pip install openai`)  
- Azure OpenAI resource with a deployed model (e.g., `gpt-4o-mini`, `gpt-4.1-mini`, ecc.)  
- Environment variables set:
  - `AZURE_OPENAI_API_KEY`  
  - `AZURE_OPENAI_ENDPOINT` (e.g., `https://<your-resource>.openai.azure.com`)  
  - `MODEL_DEPLOYMENT_NAME` (the *deployment name* of your model, not the base model id)

> ℹ️ **Endpoint note**  
> Per le Responses API su Azure, è consigliato usare `base_url = <endpoint>/openai/v1/`. Alcune preview usano ancora `api_version`, ma il percorso `/openai/v1/` è quello pensato per le funzionalità più recenti delle Responses.

## [Disclaimer by OpenAI](https://platform.openai.com/docs/assistants/migration?user-chat-app=responses)
*After achieving feature parity in the Responses API, we've deprecated the Assistants API. It will shut down on August 26, 2026. Follow the guidance below to update your integration. [Learn more](https://platform.openai.com/docs/guides/migrate-to-responses).*

# Authentication & Environment setup

In [1]:
import sys
from dotenv import load_dotenv # package python-dotenv
config_path = "../../config"
sys.path.append(config_path)
from auth import acquire_bearer_token
bearer_token, user = acquire_bearer_token(auth_method="default") # default, cli, device

if not load_dotenv(f"{config_path}/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

print("Bearer token:", bearer_token[:10], "...")
print(f'User info: {user["name"]}, upn: {user["upn"]}')
# user['raw_claims']

Environment variables have been loaded ;-)
Bearer token: eyJ0eXAiOi ...
User info: Mauro Minella, upn: mauro.minella@MngEnvMCAP883652.onmicrosoft.com


# Constants and Libraries

In [2]:
import os
from IPython.display import Markdown, display
from openai import AzureOpenAI # package openai

project_openai_endpoint = os.getenv("AIF_STD_PROJECT_OPENAI_ENDPOINT")
deployment_name=os.environ["MODEL_DEPLOYMENT_NAME"]
openai_api_version = os.getenv("AZURE_OPENAI_API_VERSION")

messages_system = "You are an AI assistant that helps people find information"
messages_user = "Tell me five good names for my new pizzeria"
follow_up_question = "Nice! Now pick the best one and explain why."

# Create a client to connecto Azure OpenAI service and deployment¶

In [3]:
# Note: Applications require bearer token auth; API keys are not supported at this time.
client = AzureOpenAI(
    azure_endpoint=project_openai_endpoint.rstrip("/") + "/",
    api_key="__unused__",  # placeholder; we will send OAuth header
    default_headers={"Authorization": f"Bearer {bearer_token}"},
    api_version = openai_api_version,
)

print(f"OpenAI Endpoint base_url: {client.base_url}")

OpenAI Endpoint base_url: https://aifv2-01-std-foundry.openai.azure.com/openai/


# First Turn
## Simple user message to start a conversation

In [4]:
response_1 = client.responses.create(
    model=os.environ["MODEL_DEPLOYMENT_NAME"],
    input=[
        {
            "role": "system",
            "content": messages_system
        },
        {
            "role": "user",
            "content": messages_user
        }
    ]
)

conversation_id = response_1.id  # <-- This is the conversation ID
print("Conversation ID:", conversation_id)

print("\nAssistant response:")
print(response_1.output[0].content[0].text)

Conversation ID: resp_09b4bb023125cda40069bc816fc5f481978944f55dbde600d0

Assistant response:
Here are five catchy names for your new pizzeria:

1. **Slice of Heaven**
2. **Crust & Craft**
3. **Dough Delight**
4. **Pizza Panorama**
5. **The Saucy Slice**

Let me know if you'd like more suggestions!


# Second Turn
## Continue the conversation by referencing the conversation_id

In [5]:
response_2 = client.responses.create(
    model=deployment_name,
    previous_response_id=response_1.id,  # <--- just for Azure OpenAI, otherwise "conversation_id" must be used
    input=follow_up_question
)

print("\nAssistant follow-up:")
print(response_2.output[0].content[0].text)


Assistant follow-up:
I’d choose **Slice of Heaven** as the best option. Here's why:

1. **Memorable**: It’s catchy and easy to remember, making it stand out.
2. **Inviting**: It evokes a sense of quality and indulgence, appealing to potential customers looking for a delightful experience.
3. **Creative**: It uses a play on words that suggests your pizzas are a heavenly treat.
4. **Versatile**: The name can work well in marketing campaigns, logos, and signage, creating a cohesive brand identity.

Overall, it combines creativity with a positive connotation that can attract food enthusiasts.
